In [58]:
from pathlib import Path
from collections import defaultdict
from itertools import combinations
import math

import numpy as np
import pandas as pd

In [59]:
# 如果当前 notebook 位于 notebooks/ 文件夹
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

DEV_DIR = PROJECT_ROOT / "data" / "processed" / "dev"

print("PROJECT_ROOT:", PROJECT_ROOT)
print("DEV_DIR:", DEV_DIR)

# 看一下 Day 1 保存了哪些文件
for file in DEV_DIR.iterdir():
    print(file.name)

TRAIN_PATH = DEV_DIR / "train.csv"
VAL_PATH = DEV_DIR / "validation.csv"
GROUND_TRUTH_PATH = DEV_DIR / "validation_ground_truth.csv"

PROJECT_ROOT: <LOCAL_PROJECT_PATH>
DEV_DIR: <LOCAL_PROJECT_PATH>\data\processed\dev
test.csv
test_ground_truth.csv
train.csv
validation.csv
validation_ground_truth.csv


In [60]:
train_df = pd.read_csv(TRAIN_PATH)
val_df = pd.read_csv(VAL_PATH)
ground_truth_df = pd.read_csv(GROUND_TRUTH_PATH)

print("Train shape:", train_df.shape)
print("Validation shape:", val_df.shape)
print("Ground Truth shape:", ground_truth_df.shape)

display(train_df.head())
display(val_df.head())
display(ground_truth_df.head())

Train shape: (724885, 5)
Validation shape: (138406, 5)
Ground Truth shape: (2540, 2)


,user_id,item_id,category_id,behavior_type,timestamp
0,100,4572582,2188684,pv,1511550516
1,100,2971043,4869428,pv,1511550600
2,100,2379198,4869428,pv,1511550806
3,100,2971043,4869428,pv,1511550885
4,100,1220136,4869428,pv,1511550908


,user_id,item_id,category_id,behavior_type,timestamp
0,100,4140845,2096639,pv,1512173626
1,100,3656511,3312150,pv,1512177480
2,100,2772937,3114694,pv,1512192261
3,100,2859169,2096639,pv,1512201298
4,1000,4599135,245312,pv,1512228047


,user_id,item_id
0,100600,4759580
1,100800,4558727
2,1008100,2700676
3,1011100,1845719
4,1012000,264220


In [61]:
print("Train users:", train_df["user_id"].nunique())
print("Validation users:", val_df["user_id"].nunique())
print("Ground Truth users:", ground_truth_df["user_id"].nunique())

print()
print("Train items:", train_df["item_id"].nunique())
print("Validation items:", val_df["item_id"].nunique())
print("Ground Truth items:", ground_truth_df["item_id"].nunique())

print()
print("Train behavior counts:")
print(train_df["behavior_type"].value_counts())

print()
print("Validation behavior counts:")
print(val_df["behavior_type"].value_counts())

print()
print("Ground Truth rows:", len(ground_truth_df))
print(
    "Users with multiple Ground Truth items:",
    (ground_truth_df["user_id"].value_counts() > 1).sum()
)

Train users: 9895
Validation users: 9719
Ground Truth users: 1760

Train items: 317721
Validation items: 92413
Ground Truth items: 2465

Train behavior counts:
behavior_type
pv      649470
cart     40241
fav      20367
buy      14807
Name: count, dtype: int64

Validation behavior counts:
behavior_type
pv      123825
cart      8095
fav       3887
buy       2599
Name: count, dtype: int64

Ground Truth rows: 2540
Users with multiple Ground Truth items: 464


In [62]:
# Train 中每个用户已经交互过的商品集合
user_history = (
    train_df
    .groupby("user_id")["item_id"]
    .apply(set)
    .to_dict()
)

# Validation Ground Truth：
# 每个用户在验证阶段真正需要预测中的商品
val_ground_truth = (
    ground_truth_df
    .groupby("user_id")["item_id"]
    .apply(set)
    .to_dict()
)

print("Users in user_history:", len(user_history))
print("Users in val_ground_truth:", len(val_ground_truth))

# 随便查看一个 Ground Truth 用户
sample_user = next(iter(val_ground_truth))

print()
print("Sample user:", sample_user)
print("Train history size:", len(user_history.get(sample_user, set())))
print("Train history sample:", list(user_history.get(sample_user, set()))[:10])
print("Ground Truth:", val_ground_truth[sample_user])

Users in user_history: 9895
Users in val_ground_truth: 1760

Sample user: 900
Train history size: 15
Train history sample: [2597702, 2426153, 3765292, 521612, 1575640, 3381872, 3551443, 2605494, 914358, 4601558]
Ground Truth: {4601558, 3624334}


In [63]:
# Train 中出现过的用户和商品
train_users = set(train_df["user_id"].unique())
train_items = set(train_df["item_id"].unique())

# Ground Truth 中的用户和商品
gt_users = set(ground_truth_df["user_id"].unique())
gt_items = set(ground_truth_df["item_id"].unique())

# Ground Truth 用户中，有多少在 Train 出现过
warm_users = gt_users & train_users
cold_users = gt_users - train_users

# Ground Truth 商品中，有多少在 Train 出现过
warm_items = gt_items & train_items
cold_items = gt_items - train_items

print("Ground Truth users:", len(gt_users))
print("Warm users:", len(warm_users))
print("Cold-start users:", len(cold_users))

print()

print("Ground Truth items:", len(gt_items))
print("Warm items:", len(warm_items))
print("Cold-start items:", len(cold_items))

print()

print(
    "Warm-user coverage:",
    len(warm_users) / len(gt_users)
)

print(
    "Warm-item coverage:",
    len(warm_items) / len(gt_items)
)

Ground Truth users: 1760
Warm users: 1760
Cold-start users: 0

Ground Truth items: 2465
Warm items: 1579
Cold-start items: 886

Warm-user coverage: 1.0
Warm-item coverage: 0.6405679513184585


In [64]:
# 如果只允许推荐 Train 中出现过的商品，
# 每个用户最多能召回多少 Ground Truth？

oracle_recalls = []

total_gt_targets = 0
warm_gt_targets = 0
cold_gt_targets = 0

for user_id, gt_items in val_ground_truth.items():

    warm_gt = gt_items & train_items
    cold_gt = gt_items - train_items

    total_gt_targets += len(gt_items)
    warm_gt_targets += len(warm_gt)
    cold_gt_targets += len(cold_gt)

    # 假设模型把所有能召回的 warm item 全部召回
    oracle_recalls.append(
        len(warm_gt) / len(gt_items)
    )

print("Total Ground Truth targets:", total_gt_targets)
print("Warm targets:", warm_gt_targets)
print("Cold-start targets:", cold_gt_targets)

print()
print(
    "Target-level warm coverage:",
    warm_gt_targets / total_gt_targets
)

print(
    "Maximum possible mean Recall:",
    np.mean(oracle_recalls)
)

Total Ground Truth targets: 2540
Warm targets: 1646
Cold-start targets: 894

Target-level warm coverage: 0.6480314960629922
Maximum possible mean Recall: 0.6542586170142988


In [65]:
item_popularity = (
    train_df
    .groupby("item_id")
    .size()
    .sort_values(ascending=False)
)

popular_items = item_popularity.index.tolist()

print("Number of train items:", len(popular_items))
print()
print("Top 10 popular items:")

display(
    item_popularity
    .head(10)
    .rename("interaction_count")
    .reset_index()
)

Number of train items: 317721

Top 10 popular items:


,item_id,interaction_count
0,3845720,231
1,812879,220
2,2331370,182
3,3371523,176
4,1591862,167
5,2367945,167
6,3031354,164
7,2338453,162
8,4211339,159
9,3715112,145


In [66]:
def recommend_popularity(
    user_id,
    k,
    popular_items
):
    # Future purchase prediction:
    # 不过滤用户过去交互过的商品

    return popular_items[:k]

In [67]:
sample_pop_recs = recommend_popularity(
    user_id=sample_user,
    k=10,
    popular_items=popular_items
)

print("Sample user:", sample_user)
print("Train history:", list(user_history.get(sample_user, set()))[:20])
print("Ground Truth:", val_ground_truth[sample_user])

print()
print("Popularity Top-10:")
print(sample_pop_recs)

print()
print(
    "Hits:",
    set(sample_pop_recs) & val_ground_truth[sample_user]
)

Sample user: 900
Train history: [2597702, 2426153, 3765292, 521612, 1575640, 3381872, 3551443, 2605494, 914358, 4601558, 33879, 209307, 1644220, 1086782, 2832319]
Ground Truth: {4601558, 3624334}

Popularity Top-10:
[3845720, 812879, 2331370, 3371523, 1591862, 2367945, 3031354, 2338453, 4211339, 3715112]

Hits: set()


In [68]:
def recall_at_k(recommendations, ground_truth, k):
    recs = set(recommendations[:k])
    gt = set(ground_truth)

    if len(gt) == 0:
        return 0.0

    hits = len(recs & gt)

    return hits / len(gt)


def hitrate_at_k(recommendations, ground_truth, k):
    recs = set(recommendations[:k])
    gt = set(ground_truth)

    if len(gt) == 0:
        return 0.0

    return float(len(recs & gt) > 0)


def ndcg_at_k(recommendations, ground_truth, k):
    gt = set(ground_truth)

    if len(gt) == 0:
        return 0.0

    dcg = 0.0

    for rank, item in enumerate(recommendations[:k], start=1):

        if item in gt:
            dcg += 1.0 / math.log2(rank + 1)

    ideal_hits = min(len(gt), k)

    idcg = sum(
        1.0 / math.log2(rank + 1)
        for rank in range(1, ideal_hits + 1)
    )

    if idcg == 0:
        return 0.0

    return dcg / idcg

In [69]:
def evaluate_model(
    recommend_func,
    ground_truth,
    ks=(10, 20, 50)
):
    results = []

    for k in ks:

        recall_scores = []
        hitrate_scores = []
        ndcg_scores = []

        for user_id, gt_items in ground_truth.items():

            recommendations = recommend_func(
                user_id=user_id,
                k=k
            )

            recall_scores.append(
                recall_at_k(
                    recommendations,
                    gt_items,
                    k
                )
            )

            hitrate_scores.append(
                hitrate_at_k(
                    recommendations,
                    gt_items,
                    k
                )
            )

            ndcg_scores.append(
                ndcg_at_k(
                    recommendations,
                    gt_items,
                    k
                )
            )

        results.append({
            "K": k,
            "Recall": np.mean(recall_scores),
            "HitRate": np.mean(hitrate_scores),
            "NDCG": np.mean(ndcg_scores)
        })

    return pd.DataFrame(results)

In [70]:
def popularity_wrapper(user_id, k):

    return recommend_popularity(
        user_id=user_id,
        k=k,
        popular_items=popular_items
    )


popularity_results = evaluate_model(
    recommend_func=popularity_wrapper,
    ground_truth=val_ground_truth,
    ks=(10, 20, 50)
)

popularity_results

,K,Recall,HitRate,NDCG
0,10,0.001989,0.002273,0.000659
1,20,0.003125,0.003409,0.000934
2,50,0.004687,0.006818,0.001332


In [71]:
history_sizes = pd.Series(
    {
        user_id: len(items)
        for user_id, items in user_history.items()
    }
)

print("User history statistics:")
print(history_sizes.describe())

print()
print("Max history size:", history_sizes.max())

print(
    "Users with > 100 unique items:",
    (history_sizes > 100).sum()
)

print(
    "Users with > 200 unique items:",
    (history_sizes > 200).sum()
)

print(
    "Users with > 500 unique items:",
    (history_sizes > 500).sum()
)

User history statistics:
count    9895.000000
mean       56.430419
std        52.315051
min         1.000000
25%        19.000000
50%        40.000000
75%        77.000000
max       429.000000
dtype: float64

Max history size: 429
Users with > 100 unique items: 1587
Users with > 200 unique items: 236
Users with > 500 unique items: 0


In [72]:
from scipy.sparse import csr_matrix

# -------------------------------------------------
# ItemCF 使用 Train 数据
# 每个 user-item 只保留一次，并保留最后一次交互时间
# -------------------------------------------------

itemcf_train = (
    train_df
    .sort_values("timestamp")
    .drop_duplicates(
        subset=["user_id", "item_id"],
        keep="last"
    )
    .copy()
)

# -------------------------------------------------
# 控制超活跃用户的历史长度
# 只保留最近 50 个 unique items
# -------------------------------------------------

MAX_ITEMCF_HISTORY = 50

itemcf_train = (
    itemcf_train
    .sort_values(["user_id", "timestamp"])
    .groupby(
        "user_id",
        group_keys=False
    )
    .tail(MAX_ITEMCF_HISTORY)
)

print("ItemCF interactions:", len(itemcf_train))
print("ItemCF users:", itemcf_train["user_id"].nunique())
print("ItemCF items:", itemcf_train["item_id"].nunique())


# -------------------------------------------------
# 建立 user / item 连续索引
# -------------------------------------------------

user_codes, user_ids = pd.factorize(
    itemcf_train["user_id"],
    sort=True
)

item_codes, item_ids = pd.factorize(
    itemcf_train["item_id"],
    sort=True
)

num_users = len(user_ids)
num_items = len(item_ids)

print()
print("Sparse matrix shape:", (num_users, num_items))


# -------------------------------------------------
# User-Item 二值稀疏矩阵
# 1 = 该用户交互过该商品
# -------------------------------------------------

X = csr_matrix(
    (
        np.ones(
            len(itemcf_train),
            dtype=np.float32
        ),
        (
            user_codes,
            item_codes
        )
    ),
    shape=(num_users, num_items)
)

print("Non-zero entries:", X.nnz)

ItemCF interactions: 341352
ItemCF users: 9895
ItemCF items: 222183

Sparse matrix shape: (9895, 222183)
Non-zero entries: 341352


In [73]:
# ItemCF 专用用户历史
itemcf_user_history = (
    itemcf_train
    .groupby("user_id")["item_id"]
    .apply(list)
    .to_dict()
)

# item_id -> sparse matrix column
item_to_col = {
    int(item_id): idx
    for idx, item_id in enumerate(item_ids)
}

# -------------------------------------------------
# 我们只需要为 Validation Ground Truth 用户
# 历史中真正会用到的 item 构建 neighbors
# -------------------------------------------------

seed_items = set()

for user_id in val_ground_truth:

    history = itemcf_user_history.get(
        user_id,
        []
    )

    seed_items.update(history)

seed_items = [
    item
    for item in seed_items
    if item in item_to_col
]

seed_cols = np.array(
    [
        item_to_col[item]
        for item in seed_items
    ],
    dtype=np.int32
)

print("Evaluation users:", len(val_ground_truth))
print("Unique seed items:", len(seed_items))

Evaluation users: 1760
Unique seed items: 52436


In [74]:
# -------------------------------------------------
# 每个 item 被多少用户交互过
# -------------------------------------------------

item_user_counts = np.asarray(
    X.sum(axis=0)
).ravel()


# -------------------------------------------------
# Seed Item × All Item 共现矩阵
#
# X[:, seed_cols].T:
#     seed_item × user
#
# X:
#     user × item
#
# 相乘后：
#     seed_item × item
# -------------------------------------------------

co_matrix = (
    X[:, seed_cols].T @ X
).tocsr()

print("Co-occurrence matrix shape:", co_matrix.shape)
print("Co-occurrence non-zero entries:", co_matrix.nnz)

Co-occurrence matrix shape: (52436, 222183)
Co-occurrence non-zero entries: 5141369


In [75]:
ITEMCF_TOP_N = 200

sim_matrix = {}

for seed_pos, seed_item in enumerate(seed_items):

    row_start = co_matrix.indptr[seed_pos]
    row_end = co_matrix.indptr[seed_pos + 1]

    candidate_cols = co_matrix.indices[
        row_start:row_end
    ]

    co_counts = co_matrix.data[
        row_start:row_end
    ]

    seed_col = seed_cols[seed_pos]

    # 排除商品自己
    mask = candidate_cols != seed_col

    candidate_cols = candidate_cols[mask]
    co_counts = co_counts[mask]

    if len(candidate_cols) == 0:
        sim_matrix[seed_item] = []
        continue

    # ItemCF cosine-style similarity
    denominator = np.sqrt(
        item_user_counts[seed_col]
        *
        item_user_counts[candidate_cols]
    )

    similarities = (
        co_counts / denominator
    )

    # -------------------------------------------------
    # 只取最相似的 Top-N
    # -------------------------------------------------

    if len(similarities) > ITEMCF_TOP_N:

        top_positions = np.argpartition(
            similarities,
            -ITEMCF_TOP_N
        )[-ITEMCF_TOP_N:]

        top_positions = top_positions[
            np.argsort(
                similarities[top_positions]
            )[::-1]
        ]

    else:

        top_positions = np.argsort(
            similarities
        )[::-1]

    neighbors = []

    for pos in top_positions:

        candidate_col = candidate_cols[pos]

        candidate_item = int(
            item_ids[candidate_col]
        )

        sim_score = float(
            similarities[pos]
        )

        neighbors.append(
            (
                candidate_item,
                sim_score
            )
        )

    sim_matrix[int(seed_item)] = neighbors


print(
    "Items with ItemCF neighbors:",
    len(sim_matrix)
)

Items with ItemCF neighbors: 52436


In [76]:
def recommend_itemcf(
    user_id,
    k,
    itemcf_user_history,
    sim_matrix
):
    # 用最近的历史 item 作为 ItemCF seed
    seed_history = itemcf_user_history.get(
        user_id,
        []
    )

    if len(seed_history) == 0:
        return []

    rank = defaultdict(float)

    # 遍历用户历史 item
    for hist_item in seed_history:

        similar_items = sim_matrix.get(
            hist_item,
            []
        )

        # 相似商品累积分数
        for candidate_item, sim_score in similar_items:

            # Future purchase prediction:
            # 不过滤历史交互商品

            rank[candidate_item] += sim_score

    # 按分数从高到低排序
    ranked_items = sorted(
        rank.items(),
        key=lambda x: x[1],
        reverse=True
    )

    return [
        item
        for item, score in ranked_items[:k]
    ]

In [77]:
sample_itemcf_recs = recommend_itemcf(
    user_id=sample_user,
    k=10,
    itemcf_user_history=itemcf_user_history,
    sim_matrix=sim_matrix
)

print("Sample user:", sample_user)

print()
print(
    "ItemCF seed history:",
    itemcf_user_history.get(
        sample_user,
        []
    )
)

print()
print(
    "Ground Truth:",
    val_ground_truth[sample_user]
)

print()
print(
    "ItemCF Top-10:",
    sample_itemcf_recs
)

print()
print(
    "Hits:",
    set(sample_itemcf_recs)
    & val_ground_truth[sample_user]
)

Sample user: 900

ItemCF seed history: [2426153, 914358, 3551443, 1644220, 3765292, 4601558, 2605494, 2832319, 2597702, 209307, 1575640, 3381872, 521612, 1086782, 33879]

Ground Truth: {4601558, 3624334}

ItemCF Top-10: [4601558, 3551443, 2832319, 3381872, 2605494, 2597702, 914358, 1644220, 1575640, 1086782]

Hits: {4601558}


In [78]:
def itemcf_wrapper(user_id, k):

    return recommend_itemcf(
        user_id=user_id,
        k=k,
        itemcf_user_history=itemcf_user_history,
        sim_matrix=sim_matrix
    )


itemcf_results = evaluate_model(
    recommend_func=itemcf_wrapper,
    ground_truth=val_ground_truth,
    ks=(10, 20, 50)
)

itemcf_results

,K,Recall,HitRate,NDCG
0,10,0.082435,0.109091,0.041902
1,20,0.143551,0.178409,0.058288
2,50,0.257444,0.303409,0.082797


In [79]:
pop_compare = popularity_results.copy()
pop_compare["Model"] = "Popularity"

itemcf_compare = itemcf_results.copy()
itemcf_compare["Model"] = "ItemCF"


baseline_results = pd.concat(
    [
        pop_compare,
        itemcf_compare
    ],
    ignore_index=True
)

baseline_results = baseline_results[
    [
        "Model",
        "K",
        "Recall",
        "HitRate",
        "NDCG"
    ]
]

baseline_results

,Model,K,Recall,HitRate,NDCG
0,Popularity,10,0.001989,0.002273,0.000659
1,Popularity,20,0.003125,0.003409,0.000934
2,Popularity,50,0.004687,0.006818,0.001332
3,ItemCF,10,0.082435,0.109091,0.041902
4,ItemCF,20,0.143551,0.178409,0.058288
5,ItemCF,50,0.257444,0.303409,0.082797


In [80]:
# -------------------------------------------------
# 检查 Ground Truth 商品与 Train 历史商品的重叠
# -------------------------------------------------

gt_total = 0

seen_before = 0
seen_pv_before = 0
seen_cart_before = 0
seen_fav_before = 0
bought_before = 0


# Train 中不同类型的历史
train_pv = (
    train_df[train_df["behavior_type"] == "pv"]
    .groupby("user_id")["item_id"]
    .apply(set)
    .to_dict()
)

train_cart = (
    train_df[train_df["behavior_type"] == "cart"]
    .groupby("user_id")["item_id"]
    .apply(set)
    .to_dict()
)

train_fav = (
    train_df[train_df["behavior_type"] == "fav"]
    .groupby("user_id")["item_id"]
    .apply(set)
    .to_dict()
)

train_buy = (
    train_df[train_df["behavior_type"] == "buy"]
    .groupby("user_id")["item_id"]
    .apply(set)
    .to_dict()
)


for user_id, gt_items in val_ground_truth.items():

    gt_total += len(gt_items)

    full_history = user_history.get(user_id, set())

    for item in gt_items:

        if item in full_history:
            seen_before += 1

        if item in train_pv.get(user_id, set()):
            seen_pv_before += 1

        if item in train_cart.get(user_id, set()):
            seen_cart_before += 1

        if item in train_fav.get(user_id, set()):
            seen_fav_before += 1

        if item in train_buy.get(user_id, set()):
            bought_before += 1


print("Total Ground Truth targets:", gt_total)

print()
print("Seen before in Train:", seen_before)
print("Seen-before rate:", seen_before / gt_total)

print()
print("PV before:", seen_pv_before)
print("Cart before:", seen_cart_before)
print("Fav before:", seen_fav_before)
print("Bought before:", bought_before)

Total Ground Truth targets: 2540

Seen before in Train: 680
Seen-before rate: 0.2677165354330709

PV before: 457
Cart before: 372
Fav before: 69
Bought before: 72


In [81]:
pop_compare = popularity_results.copy()
pop_compare["Model"] = "Popularity"

itemcf_compare = itemcf_results.copy()
itemcf_compare["Model"] = "ItemCF"

baseline_results = pd.concat(
    [
        pop_compare,
        itemcf_compare
    ],
    ignore_index=True
)

baseline_results = baseline_results[
    [
        "Model",
        "K",
        "Recall",
        "HitRate",
        "NDCG"
    ]
]

baseline_results

,Model,K,Recall,HitRate,NDCG
0,Popularity,10,0.001989,0.002273,0.000659
1,Popularity,20,0.003125,0.003409,0.000934
2,Popularity,50,0.004687,0.006818,0.001332
3,ItemCF,10,0.082435,0.109091,0.041902
4,ItemCF,20,0.143551,0.178409,0.058288
5,ItemCF,50,0.257444,0.303409,0.082797
